In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.preprocessing import load_multiple_subjects
from src.data.class_dataset import EEGClassDataset
from src.utils.logging import get_logger
from src.utils.config import (
    LATENT_DIM,
    WINDOW_SIZE,
    WINDOW_STRIDE,
    BATCH_SIZE,
    WANTED_CHANNELS,
    NUM_CHANNELS,
    LR_CLASSIFIER,
    EPOCHS_CLASSIFIER,
    NUM_LAYERS_EMBEDDER,
    NUM_LAYERS_SUPERVISOR,
    NUM_LAYERS_GENERATOR,
    NUM_LAYERS_DISCRIMINATOR,
    HIDDEN_DIM_DISCRIMINATOR,
    HIDDEN_DIM_GENERATOR,
)
from src.models.classifier import Classifier
from src.models.recovery import Recovery
from src.models.generator import Generator
from src.training.train_classifier import train_classifier

In [2]:
logger = get_logger("TSTR")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

2026-05-09 19:34:55 | INFO | Using device: cuda


In [4]:
blocks = list(range(10,13))
subj = list(range(1,11))

In [5]:
# eeg = load_subject_frequency(blocks, 12.0, "data/raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
eeg_12 = load_multiple_subjects(subj, blocks, 12.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_12[0].shape}")
logger.info(f"Mean after norm: {eeg_12[0].mean(axis=0)}")
logger.info(f"Std after norm: {eeg_12[0].std(axis=0)}")

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-010_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-011_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 528999  =      0.000 ...   528.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 582999  =      0.000 ...   582.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



...

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-010_ses-04_block-011_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 586999  =      0.000 ...   586.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-010_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 555999  =      0.000 ...   555.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
2026-05-09 19:35:32 | INFO | EEG shape: (30, 5125, 9)
2026-05-09 19:35:32 | INFO | Mean after norm: [[0.50856921 0.51923636 0.53031066 ... 0.50097069 0.50409337 0.52241926]
 [0.5096444  0.52028713 0.53071323 ... 0.49751497 0.50068058 0.51871326]
 [0.51095512 0.52112573 0.53105309 ... 0.49437302 0.49727658 0.51508845]
 ...
 [0.48366895 0.44482571 0.46037274 ... 0.42429646 0.42851896 0.42954245]
 [0.47831874 0.44471724 0.45928953 ... 0.4183641  0.42392335 0.42521432]
 [0.4731914  0.44579284 0.45816343 ... 0.41349823 0.42067453 0.42186807]]
2026-05-09 19:35:32 | INFO | Std after norm: [[0.21747101 0.22652162 0.24158509 ... 0.20499786 0.22195075 0.22069207]
 [0.22140473 0.22641778 0.2386356  ... 0.20363609 0.21814134 0.2226158 ]
 [0.22431826 0.22494869 0.23465028 ... 0.20137018 0.21308243 0.22348653]
 ...
 [0.18676841 0.19166393 0.19881719 ... 0.16832141 0.16513078 0.17801153]
 [0.18528879 0.19648031 0.19363501 ... 0.16834257 0.1

Subject: 10
Total Segments: 3


In [6]:
eeg_16 = load_multiple_subjects(subj, blocks, 16.0, "C:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\data\\raw", duration_sec=5, l_freq=10.0, h_freq=40.0, picks=WANTED_CHANNELS)
logger.info(f"EEG shape: {eeg_16[0].shape}")
logger.info(f"Mean after norm: {eeg_16[0].mean(axis=0)}")
logger.info(f"Std after norm: {eeg_16[0].std(axis=0)}")

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-010_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 525999  =      0.000 ...   525.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-011_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 528999  =      0.000 ...   528.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-001_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 582999  =      0.000 ...   582.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



...

Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-010_ses-04_block-011_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 586999  =      0.000 ...   586.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Extracting EDF parameters from C:\Users\danie_13ucdo4\OneDrive\Desktop\ITAM\Tesis\Prueba\BestTimeGAN\data\raw\sub-010_ses-04_block-012_task-ssvep_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 555999  =      0.000 ...   555.999 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1321 samples (1.321 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
2026-05-09 19:36:05 | INFO | EEG shape: (30, 5125, 9)
2026-05-09 19:36:05 | INFO | Mean after norm: [[0.46199568 0.44863044 0.48787117 ... 0.48589869 0.45047991 0.49508064]
 [0.46242449 0.45302796 0.49211406 ... 0.48914689 0.45389864 0.49675933]
 [0.46402393 0.45854059 0.497434   ... 0.49325768 0.45845046 0.499375  ]
 ...
 [0.56404878 0.58235361 0.59536412 ... 0.5788438  0.57317669 0.60290404]
 [0.56590419 0.58652293 0.60078783 ... 0.58585007 0.57634969 0.60803548]
 [0.56707381 0.59024778 0.60555618 ... 0.59321761 0.57844617 0.61273239]]
2026-05-09 19:36:05 | INFO | Std after norm: [[0.21679251 0.17953389 0.21426032 ... 0.20115698 0.19019662 0.19136643]
 [0.21736586 0.17418489 0.21560106 ... 0.19642318 0.18432727 0.19419695]
 [0.2183917  0.17127067 0.21737397 ... 0.19331116 0.17956462 0.19941944]
 ...
 [0.20574959 0.22015303 0.18628539 ... 0.19767229 0.21478738 0.19107815]
 [0.20694363 0.22653312 0.19060041 ... 0.20357805 0.2

Subject: 10
Total Segments: 3


In [7]:
dataset = EEGClassDataset(
    eeg_list=[eeg_12[0], eeg_16[0]],
    labels=[0,1],
    window_size=WINDOW_SIZE,
    hop_size=WINDOW_STRIDE,
    normalize=False,
)

In [8]:
test_dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

In [9]:
G_12 = Generator(
    z_dim=LATENT_DIM+4, 
    h_dim=HIDDEN_DIM_GENERATOR, 
    num_layers=NUM_LAYERS_GENERATOR
).to(device)

R_12 = Recovery(
    h_dim=LATENT_DIM,
    x_dim=NUM_CHANNELS,
    num_layers=2
).to(device)

G_16 = Generator(
    z_dim=LATENT_DIM+4, 
    h_dim=HIDDEN_DIM_GENERATOR, 
    num_layers=NUM_LAYERS_GENERATOR
).to(device)

R_16 = Recovery(
    h_dim=LATENT_DIM,
    x_dim=NUM_CHANNELS,
    num_layers=2
).to(device)

C = Classifier(9,512,2)

In [10]:
G_12.load_state_dict(torch.load("c:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\generator_12.3.pt", weights_only=True))
R_12.load_state_dict(torch.load("c:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\recovery_12.0.pt", weights_only=True))
G_16.load_state_dict(torch.load("c:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\generator_16.3.pt", weights_only=True))
R_16.load_state_dict(torch.load("c:\\Users\\danie_13ucdo4\\OneDrive\\Desktop\\ITAM\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\recovery_16.0.pt", weights_only=True))

<All keys matched successfully>

In [11]:
@torch.no_grad()
def generate_eeg(generator, recovery, device):
    generator.eval()
    recovery.eval()

    B = BATCH_SIZE
    T = WINDOW_SIZE
    z_dim = LATENT_DIM+4

    Z = torch.randn(B, T, z_dim, device=device)
    H_fake = generator(Z)
    X_fake = recovery(H_fake)

    X_fake = X_fake.cpu().numpy()

    return X_fake

In [12]:
@torch.no_grad()
def generate_dataset(generator, recovery, n_batches, device):
    """
    Generate synthetic EEG dataset.

    Returns
    -------
    np.ndarray of shape [N_trials, T, C]
    """
    generator.eval()
    all_samples = []

    for _ in range(n_batches):
        x_fake = generate_eeg(generator, recovery, device)
        all_samples.append(x_fake)

    # Concatenate along batch dimension
    data = np.concatenate(all_samples, axis=0)  # [N_total, 512, 9]

    return data

In [13]:
syn_12 = generate_dataset(G_12, R_12, 300, device)
syn_16 = generate_dataset(G_16, R_16, 300, device)

In [14]:
syn_dataset = EEGClassDataset(
    eeg_list=[syn_12, syn_16],
    labels=[0,1],
    window_size=WINDOW_SIZE,
    hop_size=WINDOW_SIZE,
    normalize=False,
)

In [15]:
train_data, val_data = random_split(syn_dataset, [0.87,0.13])

In [16]:
len(syn_dataset)

9600

In [17]:
train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [18]:
val_dataloader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [19]:
train_classifier(
    C,
    train_dataloader,
    val_dataloader,
    10,
    LR_CLASSIFIER,
    device,
    logger
)

2026-05-09 19:36:11 | INFO | Epoch 1 | Train Loss: 0.1249 | Train Acc: 0.9557 | Val Loss: 0.0375 | Val Acc: 0.9896
2026-05-09 19:36:13 | INFO | Epoch 2 | Train Loss: 0.0248 | Train Acc: 0.9916 | Val Loss: 0.0109 | Val Acc: 0.9968
2026-05-09 19:36:14 | INFO | Epoch 3 | Train Loss: 0.0185 | Train Acc: 0.9939 | Val Loss: 0.0084 | Val Acc: 0.9968
2026-05-09 19:36:15 | INFO | Epoch 4 | Train Loss: 0.0149 | Train Acc: 0.9952 | Val Loss: 0.0228 | Val Acc: 0.9944
2026-05-09 19:36:17 | INFO | Epoch 5 | Train Loss: 0.0141 | Train Acc: 0.9952 | Val Loss: 0.0025 | Val Acc: 1.0000
2026-05-09 19:36:18 | INFO | Epoch 6 | Train Loss: 0.0213 | Train Acc: 0.9927 | Val Loss: 0.0108 | Val Acc: 0.9968
2026-05-09 19:36:19 | INFO | Epoch 7 | Train Loss: 0.0103 | Train Acc: 0.9966 | Val Loss: 0.0051 | Val Acc: 0.9984
2026-05-09 19:36:20 | INFO | Epoch 8 | Train Loss: 0.0144 | Train Acc: 0.9949 | Val Loss: 0.0002 | Val Acc: 1.0000
2026-05-09 19:36:21 | INFO | Epoch 9 | Train Loss: 0.0137 | Train Acc: 0.9955 | 

Classifier(
  (temporal): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
  (spatial): Conv2d(8, 16, kernel_size=(9, 1), stride=(1, 1), groups=8, bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (dropout): Dropout(p=0.25, inplace=False)
  (sep): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [20]:
torch.save(C.state_dict(), "TSClassifier3.pt")

In [21]:
C.load_state_dict(torch.load("TSClassifier3.pt", weights_only=True))

<All keys matched successfully>

In [22]:
def evaluate(model, dataloader, device):
    model.to(device)
    model.eval()  # switch to inference mode


    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():  # disable gradients
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)              # (B, num_classes)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())

    accuracy = correct / total

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return accuracy, all_preds, all_labels

In [23]:
accuracy, all_preds, all_labels = evaluate(C, test_dataloader, device)

In [24]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.51      0.33      0.40      1110
           1       0.50      0.68      0.57      1098

    accuracy                           0.50      2208
   macro avg       0.50      0.50      0.49      2208
weighted avg       0.50      0.50      0.49      2208

